<a href="https://colab.research.google.com/github/ishan-bharath/MIA-AI-Therapist-/blob/main/MIA_GPT2_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece


In [ ]:
from google.colab import files
import pandas as pd

# Upload your CSV file
print("Click 'Choose Files' and select therapy_data.csv from your computer:")
uploaded = files.upload()

# Now the file is in Colab's /content/ folder
data_path = "therapy_data.csv"

print("✓ File uploaded successfully!")


Click 'Choose Files' and select therapy_data.csv from your computer:


Saving therapy_data.csv to therapy_data.csv
✓ File uploaded successfully!


In [ ]:
import pandas as pd

df = pd.read_csv(data_path)

print("Columns:", df.columns.tolist())
print("Total rows:", len(df))
print(df.head(3))

# Basic cleaning: drop rows with missing values
df = df.dropna(subset=["input", "output"])
print("Rows after dropping NaNs:", len(df))

# Build a single text field for GPT-2
def build_conversation(row):
    user = str(row["input"]).strip()
    mia = str(row["output"]).strip()
    # Format as conversation
    return f"User: {user}\nMIA: {mia}\n\n"

df["text"] = df.apply(build_conversation, axis=1)

# Quick sanity check
print("\nExample formatted conversation:\n")
print(df["text"].iloc[0])


Columns: ['input', 'output']
Total rows: 30
                                               input  \
0  I'm having trouble sleeping because my mind wo...   
1  I hate this new motherfucking boss at my job b...   
2  Dude I cheated on my girlfriend bro. I felt it...   

                                              output  
0  That sounds exhausting man. I hate it when tha...  
1  Yeah screw him dude. That nigga sounds really ...  
2  Bro that's wrong and you know it. What's going...  
Rows after dropping NaNs: 30

Example formatted conversation:

User: I'm having trouble sleeping because my mind won't shut off and I'm anxious about everything
MIA: That sounds exhausting man. I hate it when that happens. You wanna tell me what's on your mind that's keeping you up?




In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

# Create HF Dataset from pandas
dataset = Dataset.from_pandas(df[["text"]])

# Split into train/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print("Train examples:", len(train_dataset))
print("Eval examples:", len(eval_dataset))

# Load GPT-2 tokenizer
model_name = "gpt2"  # small, good for Colab
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 has no pad token; we set it to eos token
tokenizer.pad_token = tokenizer.eos_token

max_length = 256  # enough for your short dialogues

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print("✓ Tokenization complete!")
print("\nFirst tokenized example (first 10 tokens):")
print(tokenized_train[0]["input_ids"][:10])



Train examples: 27
Eval examples: 3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

✓ Tokenization complete!

First tokenized example (first 10 tokens):
[12982, 25, 314, 1394, 4953, 284, 407, 1337, 546, 607]


In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
import os

model = AutoModelForCausalLM.from_pretrained(model_name)

# Align padding token ids
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.eos_token_id

# Create output directory for saving models
model_output_dir = "/content/mia_gpt2_model"
os.makedirs(model_output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=model_output_dir,
    num_train_epochs=3,          # 3 epochs for 30 examples
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",       # changed from evaluation_strategy
    save_strategy="epoch",
    logging_steps=5,
    learning_rate=5e-5,
    warmup_steps=10,
    weight_decay=0.01,
    fp16=True,                   # use GPU half precision
    report_to="none",            # disable wandb
)

print("✓ Model loaded and training args configured!")
print(f"Model output directory: {model_output_dir}")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded and training args configured!
Model output directory: /content/mia_gpt2_model


In [ ]:
import torch

# Labels are the same as input_ids for language modeling
def to_lm_labels(batch):
    batch["labels"] = batch["input_ids"].copy()
    return batch

tokenized_train_lm = tokenized_train.map(to_lm_labels, batched=True)
tokenized_eval_lm = tokenized_eval.map(to_lm_labels, batched=True)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_lm,
    eval_dataset=tokenized_eval_lm,
)

print("Starting training…")
trainer.train()
print("\n✓ Training complete!")


Map:   0%|          | 0/27 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Starting training…


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,8.486862,1.843845
2,2.676783,0.547156
3,0.568595,0.482551


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Training complete!


In [ ]:
# Save final model and tokenizer
save_path = model_output_dir + "/final"
import os
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("✓ Model and tokenizer saved!")
print(f"Location: {save_path}")

# List files
import os
saved_files = os.listdir(save_path)
print("\nFiles saved:")
for f in saved_files:
    print(f"  - {f}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model and tokenizer saved!
Location: /content/mia_gpt2_model/final

Files saved:
  - generation_config.json
  - tokenizer_config.json
  - model.safetensors
  - config.json
  - tokenizer.json


In [ ]:
from transformers import pipeline
import torch

# Reload from saved directory
chat_model = AutoModelForCausalLM.from_pretrained(save_path)
chat_tokenizer = AutoTokenizer.from_pretrained(save_path)
chat_tokenizer.pad_token = chat_tokenizer.eos_token

generator = pipeline(
    "text-generation",
    model=chat_model,
    tokenizer=chat_tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

def chat_with_mia(user_message, max_new_tokens=60, temperature=0.7, top_p=0.9):
    prompt = f"User: {user_message}\nMIA:"
    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        num_return_sequences=1,
        pad_token_id=chat_tokenizer.eos_token_id,
    )
    full_text = outputs[0]["generated_text"]

    # Extract only MIA's part after "MIA:"
    if "MIA:" in full_text:
        mia_part = full_text.split("MIA:", 1)[1]
    else:
        mia_part = full_text

    # Stop if it starts a new "User:" segment
    mia_part = mia_part.split("User:", 1)[0]
    return mia_part.strip()

# Test with a few prompts
test_prompts = [
    "I'm feeling really lonely tonight",
    "I messed up at work today",
    "I'm anxious about my future",
]

print("=" * 60)
print("Testing MIA GPT-2 Model")
print("=" * 60)

for p in test_prompts:
    print(f"\nUser: {p}")
    response = chat_with_mia(p)
    print(f"MIA: {response}")
    print("-" * 60)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'pad_token_id', 'temperature', 'num_return_sequences', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing MIA GPT-2 Model

User: I'm feeling really lonely tonight


Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MIA: I'm not sure what's wrong with me, but I feel like I'm in the wrong place.
------------------------------------------------------------

User: I messed up at work today


Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MIA: I'm not sure how I did it. I think that's why I went through this. I'm not sure how I did it. I think that's why I came back and I'm not sure how I did it.
------------------------------------------------------------

User: I'm anxious about my future
MIA: You're right. I'm worried about the future.
------------------------------------------------------------
